# Merge Data Explorer

Sanity-check merge data for **one run**. Verify labels, inspect exemplar crops,
and understand the distance-based features that drive merge decisions.

Run this before the ML notebook to make sure the data pipeline produces valid inputs.

**Spec**: `specs/016-merge-eda-validation/spec.md`

**Key principle**: The merge decision is fundamentally about **distance** — are these
two clusters close enough to be the same person? Features should measure the
relationship between the two clusters, not properties of the dataset.

**Empirical observation**: `p10_cross_dist ≈ 0.60` appears to be the key separator.
Pairs where the 10th-percentile cross-cluster distance is below ~0.60 have strong
overlap evidence and tend to be merges. P10 is robust because it ignores the noisiest
90% of face pairings (bad poses, occlusions) and focuses on the closest subset.

**Core features** (distance-based — measure pair relationship, not dataset properties):
- `min_exemplar_dist` — best-case distance between representative faces
- `min_cross_dist` — absolute closest pair across clusters
- `p10_cross_dist` — near-side of cross-cluster distribution (~0.60 empirical threshold)
- `p50_cross_dist` — median cross-cluster distance
- `support_fraction` — fraction of cross-pairs close (breadth of evidence)
- `post_merge_diameter` — how big the merged cluster would be
- `diameter_expansion` — relative growth if merged
- `size_a/b`, `size_ratio` — cluster sizes (large-cluster merges need stronger evidence)
- `mean_intra_dist_a/b` — intra-cluster compactness (context for inter-cluster distance)
- `same_image_min_dist` — anti-merge: same photo → different people

**Derived ratio features** (no backend changes — computed from above):
- `p10_over_p50` — near-tail / median: low ratio = distribution right-skewed = few close pairs
- `inter_over_intra` — p10 / max_intra_dist: the margin signal; <1.0 means clusters overlap

In [ ]:
import hashlib
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

from face_cluster.loader import load_pipeline_result
from face_cluster.features import FeatureComputer, MergeFeatureContext

%matplotlib inline
plt.rcParams["figure.figsize"] = (14, 5)

# --- Project root (two levels up from notebooks/face_clustering/) ---
PROJECT_ROOT = Path("../../")
RESULTS_DIR = PROJECT_ROOT / "results"

In [ ]:
# --- Canonical runs: one per source dataset ---
# Face IDs are identical across all runs within a dataset (verified).
# Pick the run with the most merge activity; use crops from a sibling run if needed.
CANONICAL_RUNS = {
    "Google_Germany": {
        "run": "Germany_18_recluster_103838",
        "crop_source": "Germany_10",
        "faces": 1103, "images": 534,
    },
    "Austria_24": {
        "run": "Austria24_4_recluster_005203",
        "crop_source": "Austria24_4",
        "faces": 753, "images": 369,
    },
    "Noa_5-7": {
        "run": "Budapest_1_recluster_111551",
        "crop_source": "Noa5-7",
        "faces": 1206, "images": 374,
    },
    "Noa_2-5": {
        "run": "Noa2-5_recluster_014740",
        "crop_source": "Noa2-5",
        "faces": 418, "images": 177,
    },
}

print("Canonical runs (one per source dataset):")
for ds, info in CANONICAL_RUNS.items():
    run_dir = RESULTS_DIR / info["run"]
    crop_dir = RESULTS_DIR / info["crop_source"] / "crops"
    ml_path = run_dir / "merge_log.json"
    n_merged = 0
    if ml_path.exists():
        ml = json.loads(ml_path.read_text())
        n_merged = sum(1 for e in ml if e["action"] == "merged")
    print(f"  {ds:20s}  run={info['run']:55s}  merges={n_merged}  "
          f"crops={'OK' if crop_dir.exists() else 'MISSING'}")

## 1. Pick a dataset to inspect

Select one of the canonical runs. Each represents a unique source album.

In [ ]:
# --- Pick a dataset ---
DATASET = "Google_Germany"  # Change to inspect a different dataset

info = CANONICAL_RUNS[DATASET]
RUN_DIR = RESULTS_DIR / info["run"]
CROP_DIR = RESULTS_DIR / info["crop_source"] / "crops"

# Pipeline default candidate threshold — matches face_cluster/config.py
CANDIDATE_THRESHOLD = 0.45

result = load_pipeline_result(RUN_DIR)
cr = result.cluster_result
merge_log = json.loads((RUN_DIR / "merge_log.json").read_text())

n_merged = sum(1 for e in merge_log if e["action"] == "merged")
n_rejected = sum(1 for e in merge_log if e["action"] == "rejected")

print(f"Dataset: {DATASET}")
print(f"Run: {RUN_DIR.name}")
print(f"Crop source: {CROP_DIR}")
print(f"Faces: {len(result.faces)}  Clusters: {cr.n_clusters}  Noise: {cr.n_noise}")
print(f"Merge log: {len(merge_log)} entries  (merged={n_merged}  rejected={n_rejected})")

if n_merged == 0:
    print("\nWARNING: This run has no merges. All labels are heuristic rejects.")
    print("Datasets with merges:", [ds for ds, i in CANONICAL_RUNS.items()
          if json.loads((RESULTS_DIR / i["run"] / "merge_log.json").read_text())
          and sum(1 for e in json.loads((RESULTS_DIR / i["run"] / "merge_log.json").read_text())
                  if e["action"] == "merged") > 0])

## 2. Merge log table + transitivity analysis

The merge log records decisions iteratively. When A merges with B, cluster B is absorbed
into A. If A then merges with C, the log says `(A, C)` — but the feature matrix was
computed on the **pre-merge** state where B and C were separate clusters.

We use **union-find** to compute the transitive closure: if A+B merged and A+C merged,
then B and C are also in the same identity group and should be labeled positive.

In [ ]:
# Show merge log as a table
cols = ["cluster_a", "cluster_b", "cluster_a_size", "cluster_b_size",
        "exemplar_dist", "action", "passes_exemplar", "passes_support",
        "passes_margin", "passes_diameter", "rejection_reason"]
df_log = pd.DataFrame(merge_log)[[c for c in cols if c in pd.DataFrame(merge_log).columns]]
display(df_log)

# --- Union-find for transitive closure of merges ---
class UnionFind:
    def __init__(self):
        self.parent = {}
    def find(self, x):
        self.parent.setdefault(x, x)
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.parent[ra] = rb

uf = UnionFind()
for entry in merge_log:
    if entry["action"] == "merged":
        uf.union(entry["cluster_a"], entry["cluster_b"])

# Build set of all cluster IDs that belong to the same merged group
merged_groups = {}
for entry in merge_log:
    for cid in [entry["cluster_a"], entry["cluster_b"]]:
        root = uf.find(cid)
        merged_groups.setdefault(root, set()).add(cid)

# A pair should be labeled positive if both clusters end up in the same group
def is_same_identity(ca, cb):
    return uf.find(ca) == uf.find(cb)

print(f"\nMerged identity groups (transitive closure):")
for root, members in merged_groups.items():
    if len(members) > 1:
        print(f"  Group {root}: clusters {sorted(members)}")

# Build distance matrix and compute features
embs = []
for f in result.faces:
    e = f.embedding_normalized if f.embedding_normalized is not None else f.embedding
    if e is not None:
        embs.append(e / (np.linalg.norm(e) + 1e-9))
    else:
        embs.append(np.zeros(512, dtype=np.float32))
mat = np.stack(embs).astype(np.float32)
dist_mat = np.clip(1.0 - mat @ mat.T, 0.0, 2.0)

ctx = MergeFeatureContext(
    cluster_result=cr, faces=result.faces,
    distance_matrix=dist_mat, graph_result=None,
)
fc = FeatureComputer()
pair_features = fc.compute_all_pairs(ctx, candidate_threshold=CANDIDATE_THRESHOLD)
df = fc.to_dataframe(pair_features)

# Label using transitive closure (not just direct merge log pairs)
df["label"] = df.apply(
    lambda r: 1 if is_same_identity(int(r["cluster_a"]), int(r["cluster_b"])) else 0,
    axis=1,
)

# Compare: how many labels differ from naive (direct merge log) approach?
merged_pairs_direct = set()
for entry in merge_log:
    if entry["action"] == "merged":
        a, b = entry["cluster_a"], entry["cluster_b"]
        merged_pairs_direct.add((min(a, b), max(a, b)))
df["label_naive"] = df.apply(
    lambda r: 1 if (int(r["cluster_a"]), int(r["cluster_b"])) in merged_pairs_direct else 0,
    axis=1,
)
n_transitive_fixes = int((df["label"] != df["label_naive"]).sum())
if n_transitive_fixes > 0:
    print(f"\nTransitivity fix: {n_transitive_fixes} pairs relabeled "
          f"(were 0 by naive, now 1 by transitive closure)")
    display(df[df["label"] != df["label_naive"]][["cluster_a", "cluster_b", "min_exemplar_dist", "label_naive", "label"]])
else:
    print(f"\nNo transitivity corrections needed for this run.")
df.drop(columns=["label_naive"], inplace=True)

print(f"\nCandidate pairs: {len(df)}")
print(f"Label distribution: {df['label'].value_counts().to_dict()}")
print(f"Feature columns: {len(df.columns) - 3}")
nan_counts = df.isnull().sum().sort_values(ascending=False)
nans = nan_counts[nan_counts > 0]
if len(nans) > 0:
    print(f"NaN counts (top 5):")
    print(nans.head())

## 3. Show exemplar crops for merged and rejected pairs

Visual check: do merged pairs look like the same person? Do rejected pairs look different?

In [ ]:
def load_crop(face_id, run_dir, fallback_crop_dir=None):
    """Load a face crop image. Try run_dir/crops first, then fallback."""
    dirs_to_try = [Path(run_dir) / "crops"]
    if fallback_crop_dir:
        dirs_to_try.append(Path(fallback_crop_dir))
    for crop_dir in dirs_to_try:
        for pattern in [f"face_{face_id:04d}_aligned.jpg", f"face_{face_id:04d}.jpg",
                        f"face_{face_id:04d}_aligned.png", f"face_{face_id:04d}.png"]:
            p = crop_dir / pattern
            if p.exists():
                return Image.open(p).convert("RGB")
    return None


def show_pair(ca, cb, label, ex_dist, run_dir, crop_dir, cr, faces):
    """Show exemplar crops for a cluster pair side by side."""
    ex_a = cr.exemplars.get(ca, cr.clusters.get(ca, [])[:3])
    ex_b = cr.exemplars.get(cb, cr.clusters.get(cb, [])[:3])
    fids_a = [faces[i].face_id for i in ex_a[:3]]
    fids_b = [faces[i].face_id for i in ex_b[:3]]

    all_fids = fids_a + fids_b
    n = len(all_fids)
    fig, axes = plt.subplots(1, n, figsize=(2.5 * n, 2.5))
    if n == 1:
        axes = [axes]
    for i, fid in enumerate(all_fids):
        img = load_crop(fid, run_dir, crop_dir)
        if img:
            axes[i].imshow(img)
        else:
            axes[i].text(0.5, 0.5,
                         f"face_{fid:04d}\nmissing",
                         ha="center", va="center", fontsize=7, color="gray")
        side = f"C{ca}" if i < len(fids_a) else f"C{cb}"
        axes[i].set_title(f"{side} / {fid}", fontsize=9)
        axes[i].axis("off")
    tag = "MERGED" if label == 1 else "REJECT"
    color = "green" if label == 1 else "red"
    fig.suptitle(f"{tag}: C{ca} vs C{cb}  (exemplar_dist={ex_dist:.3f})",
                 fontsize=11, color=color)
    fig.tight_layout()
    plt.show()


# Verify crop path
if CROP_DIR.exists():
    n_crops = len(list(CROP_DIR.glob("face_*")))
    print(f"Crop directory: {CROP_DIR.resolve()} ({n_crops} files)")
else:
    print(f"WARNING: Crop directory not found at {CROP_DIR.resolve()}")

# Show all merged pairs
if df[df["label"] == 1].empty:
    print("No merged pairs to display.")
else:
    print("=== MERGED PAIRS ===")
    for _, row in df[df["label"] == 1].iterrows():
        show_pair(int(row["cluster_a"]), int(row["cluster_b"]),
                  1, row["min_exemplar_dist"], RUN_DIR, CROP_DIR, cr, result.faces)

# Show rejected pairs (closest ones -- borderline cases most likely to be wrong)
print("\n=== REJECTED PAIRS (closest 5 by exemplar dist) ===")
rejected = df[df["label"] == 0].nsmallest(5, "min_exemplar_dist")
for _, row in rejected.iterrows():
    show_pair(int(row["cluster_a"]), int(row["cluster_b"]),
              0, row["min_exemplar_dist"], RUN_DIR, CROP_DIR, cr, result.faces)

## 4. Distance Feature Analysis

The merge question is simple: **are these two clusters the same person?**

The features that answer this are all distance-based — they measure the relationship
between the two clusters, or what happens if we merge them.

Features like `t_local` and `t_global` measure cluster spread (a property of the
dataset / clustering algorithm), not the pair relationship. They are excluded from
the core feature set.

In [ ]:
# Features that measure the relationship between two clusters.
# Grouped by what they capture; all vary per pair (not per run).
DISTANCE_FEATURES = [
    # Cross-cluster distance distribution
    "min_exemplar_dist",      # best-case distance between representative faces
    "exemplar_dist_mean",     # average exemplar-to-exemplar distance
    "min_cross_dist",         # absolute closest pair across clusters
    "p10_cross_dist",         # near-side (~0.60 empirical threshold)
    "p25_cross_dist",         # lower quartile
    "p50_cross_dist",         # median cross-cluster distance
    "cross_dist_iqr",         # spread of the distribution
    "support_fraction",       # fraction of cross-pairs below threshold (breadth)
    # Merge consequences
    "post_merge_diameter",    # how big the merged cluster would be
    "diameter_expansion",     # relative growth: post_merge / max(dia_a, dia_b)
    # Cluster geometry — vary per pair, not per run
    "size_a",                 # faces in cluster A
    "size_b",                 # faces in cluster B
    "size_ratio",             # size_min / size_max — asymmetry
    "mean_intra_dist_a",      # intra-cluster compactness A
    "mean_intra_dist_b",      # intra-cluster compactness B
    # Anti-merge signal
    "same_image_min_dist",    # same source photo -> different people
]
available_dist = [f for f in DISTANCE_FEATURES if f in df.columns]

# --- Derived ratio features (no backend changes needed) ---
# p10_over_p50: near-tail / median — is the distribution left-heavy (many close pairs)?
#   Low ratio = heavy near-tail = strong overlap evidence
df["p10_over_p50"] = (
    df["p10_cross_dist"] / df["p50_cross_dist"].replace(0, np.nan)
).clip(0, 2)

# inter_over_intra: p10 / max intra-cluster spread — the margin signal.
#   Ratio < 1.0 means inter-cluster distance < intra-cluster spread -> strong overlap.
#   This is the "are they closer to each other than to themselves?" test.
df["intra_max"] = df[["mean_intra_dist_a", "mean_intra_dist_b"]].max(axis=1)
df["inter_over_intra"] = (
    df["p10_cross_dist"] / df["intra_max"].replace(0, np.nan)
).clip(0, 5)

derived = ["p10_over_p50", "inter_over_intra"]
available_dist += [d for d in derived if d in df.columns]

print(f"Features ({len(available_dist)} total):")
for f in available_dist:
    print(f"  {f}")

missing = [f for f in DISTANCE_FEATURES if f not in df.columns]
if missing:
    print(f"\nMissing from dataframe: {missing}")

print(f"\nMedians by label:")
display(df.groupby("label")[available_dist].median().T.rename(columns={0: "Reject", 1: "Merge"}))

In [ ]:
# Distance feature distributions by label
n_feats = len(available_dist)
n_cols = min(4, n_feats)
n_rows = (n_feats + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
axes_flat = axes.flat if n_feats > 1 else [axes]
for ax, feat in zip(axes_flat, available_dist):
    for label, color, name in [(1, "green", "Merge"), (0, "red", "Reject")]:
        vals = df[df["label"] == label][feat].dropna()
        if len(vals) > 0:
            ax.hist(vals, bins=25, alpha=0.5, color=color, label=name, density=True)
    ax.set_title(feat, fontweight="bold", fontsize=10)
    ax.legend(fontsize=8)
# Hide empty axes
for ax in list(axes_flat)[len(available_dist):]:
    ax.set_visible(False)
fig.suptitle("Distance Feature Distributions: Merge vs Reject", fontsize=13)
fig.tight_layout()
plt.show()

In [ ]:
# --- p10 threshold analysis ---
# Empirical observation: p10_cross_dist ~ 0.60 separates merges from rejects.
# This cell evaluates it as a standalone classifier and sweeps for the optimal cutoff.
from sklearn.metrics import f1_score, classification_report

P10_THRESHOLD = 0.60

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: p10 distribution by label with threshold marked
ax = axes[0]
for label, color, name in [(1, "green", "Merge"), (0, "red", "Reject")]:
    vals = df[df["label"] == label]["p10_cross_dist"].dropna()
    if len(vals):
        ax.hist(vals, bins=25, alpha=0.55, color=color,
                label=f"{name} (n={len(vals)})", density=True)
ax.axvline(P10_THRESHOLD, color="black", linestyle="--", linewidth=1.5,
           label=f"p10 = {P10_THRESHOLD}")
ax.set_xlabel("p10_cross_dist")
ax.set_title(f"p10 distribution by label  (threshold ~ {P10_THRESHOLD})")
ax.legend()

# Right: inter_over_intra — margin signal
ax = axes[1]
if "inter_over_intra" in df.columns:
    for label, color, name in [(1, "green", "Merge"), (0, "red", "Reject")]:
        vals = df[df["label"] == label]["inter_over_intra"].dropna()
        if len(vals):
            ax.hist(vals, bins=25, alpha=0.55, color=color,
                    label=f"{name} (n={len(vals)})", density=True)
    ax.axvline(1.0, color="black", linestyle="--", linewidth=1.5,
               label="inter = intra (boundary)")
    ax.set_xlabel("inter_over_intra  (p10 / max_intra_dist)")
    ax.set_title("Margin: clusters overlap when ratio < 1.0")
    ax.legend()

fig.tight_layout()
plt.show()

# Evaluate p10 threshold as a simple classifier
if df["label"].sum() > 0:
    y_p10 = (df["p10_cross_dist"] < P10_THRESHOLD).astype(int)
    print(f"p10 < {P10_THRESHOLD} as a classifier:")
    print(classification_report(df["label"], y_p10,
                                target_names=["Reject", "Merge"], zero_division=0))

    # Sweep to find the empirically optimal p10 threshold for this run
    best_f1, best_t = 0.0, P10_THRESHOLD
    for t in np.arange(0.20, 1.00, 0.02):
        f = f1_score(df["label"], (df["p10_cross_dist"] < t).astype(int), zero_division=0)
        if f > best_f1:
            best_f1, best_t = f, t
    print(f"Optimal p10 threshold for this run: {best_t:.2f}  (F1={best_f1:.3f})")

In [ ]:
# Scatter: min_exemplar_dist vs post_merge_diameter
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for label, color, marker, name in [(0, "red", "x", "Reject"), (1, "green", "o", "Merge")]:
    sub = df[df["label"] == label]
    axes[0].scatter(sub["min_exemplar_dist"], sub["post_merge_diameter"],
                    c=color, marker=marker, alpha=0.6, label=name, s=50)
axes[0].set_xlabel("min_exemplar_dist")
axes[0].set_ylabel("post_merge_diameter")
axes[0].set_title("Exemplar distance vs merged cluster size")
axes[0].legend()

# Scatter: min_exemplar_dist vs support_fraction
for label, color, marker, name in [(0, "red", "x", "Reject"), (1, "green", "o", "Merge")]:
    sub = df[df["label"] == label]
    axes[1].scatter(sub["min_exemplar_dist"], sub["support_fraction"],
                    c=color, marker=marker, alpha=0.6, label=name, s=50)
axes[1].set_xlabel("min_exemplar_dist")
axes[1].set_ylabel("support_fraction")
axes[1].set_title("Exemplar distance vs breadth of evidence")
axes[1].legend()

fig.tight_layout()
plt.show()

## 5. Distance Feature Correlation

Which distance features are redundant? Which correlate most with the label?

In [ ]:
# Correlation with label
corr_label = df[available_dist + ["label"]].corr()["label"].drop("label").abs().sort_values(ascending=False)
print("Distance features ranked by |correlation with label|:")
display(corr_label.to_frame("abs_corr"))

# Pairwise correlation heatmap
fig, ax = plt.subplots(figsize=(10, 8))
cm = df[available_dist].corr()
im = ax.imshow(cm, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(len(available_dist)))
ax.set_yticks(range(len(available_dist)))
ax.set_xticklabels(available_dist, rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(available_dist, fontsize=9)
fig.colorbar(im, ax=ax, shrink=0.8)
ax.set_title("Distance Feature Correlation")
fig.tight_layout()
plt.show()

# Flag redundant pairs
print("\nHighly correlated pairs (|r| > 0.9) — candidates for removal:")
for i, f1 in enumerate(available_dist):
    for f2 in available_dist[i+1:]:
        r = cm.loc[f1, f2]
        if abs(r) > 0.9:
            print(f"  {f1} <-> {f2}: r={r:.3f}")

## 6. Negative Label Harvesting

Non-candidate cluster pairs (exemplar distance > threshold) from the same run are
strong negative labels. We verify they don't contradict transitivity.

In [ ]:
# Harvest non-candidate pairs as strong negatives
cluster_ids = sorted(cr.clusters.keys())
non_candidate_pairs = []
for i, cid_a in enumerate(cluster_ids):
    for cid_b in cluster_ids[i + 1:]:
        ex_a = cr.exemplars.get(cid_a, cr.clusters[cid_a])
        ex_b = cr.exemplars.get(cid_b, cr.clusters[cid_b])
        min_dist = float(dist_mat[np.ix_(ex_a, ex_b)].min())
        if min_dist > CANDIDATE_THRESHOLD:
            # Verify not transitively merged
            if not is_same_identity(cid_a, cid_b):
                non_candidate_pairs.append({
                    "cluster_a": cid_a,
                    "cluster_b": cid_b,
                    "min_exemplar_dist": min_dist,
                    "source": "auto_negative",
                })

print(f"Non-candidate pairs available as strong negatives: {len(non_candidate_pairs)}")
print(f"Candidate pairs in feature df: {len(df)}")
print(f"  of which positive (merge): {(df['label'] == 1).sum()}")
print(f"  of which negative (reject): {(df['label'] == 0).sum()}")

if non_candidate_pairs:
    df_nc = pd.DataFrame(non_candidate_pairs)
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(df_nc["min_exemplar_dist"], bins=30, color="gray", alpha=0.7,
            label=f"Non-candidates (n={len(df_nc)})")
    ax.hist(df[df["label"] == 0]["min_exemplar_dist"], bins=30, color="red", alpha=0.5,
            label=f"Rejected candidates (n={(df['label']==0).sum()})")
    ax.hist(df[df["label"] == 1]["min_exemplar_dist"], bins=30, color="green", alpha=0.5,
            label=f"Merged (n={(df['label']==1).sum()})")
    ax.set_xlabel("min_exemplar_dist")
    ax.set_title("Distance distribution: merged vs rejected vs non-candidates")
    ax.legend()
    fig.tight_layout()
    plt.show()